# Session 1: OpenAI API Engineering with Structured Outputs (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 1. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks. Use this as your reference during class delivery.

**McKinsey Context:** All demos and tasks use real-world consulting scenarios — strategy engagements, M&A due diligence, digital transformation assessments, market sizing exercises, and CEO briefings — to demonstrate how structured outputs power consulting workflows at scale.

## Learning Objectives

By the end of this session, students will be able to:

1. **Use streaming** and track token usage for cost management
2. **Generate structured JSON** outputs using `response_format` parameter
3. **Define and use function calling** (tool use) with the OpenAI API
4. **Validate LLM outputs** with Pydantic models for type safety
5. **Build extraction pipelines** that turn unstructured text into structured data
6. **Implement robust API wrappers** with retries and error handling

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import os
import time
from pydantic import BaseModel, Field
from typing import Optional, List

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: OpenAI API Deep Dive — Streaming, Token Usage, and Finish Reasons

In production consulting systems you need to: (a) track token usage for engagement cost management, (b) stream responses for real-time partner briefings, and (c) inspect finish reasons to know if the model completed its analysis or was cut off.

**McKinsey Scenario:** A Partner asks the AI assistant to generate a concise market overview for a CEO briefing. We track tokens to manage costs across multiple client engagements and stream the response so the delivery team can review output in real time.

In [ ]:
# Demo 1 - OpenAI API Deep Dive (McKinsey Consulting Context)

client = openai.OpenAI()

# Part A: Token usage tracking — generating a market overview for a CEO briefing
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Provide a 3-sentence executive summary of the global healthcare market outlook for a McKinsey CEO briefing."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150"))
)

print("=== Token Usage (Engagement Cost Tracking) ===")
print(f"Prompt tokens    : {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens     : {response.usage.total_tokens}")
print(f"Finish reason    : {response.choices[0].finish_reason}")
print(f"\nCEO Briefing Response:\n{response.choices[0].message.content}")

# Part B: Streaming responses — real-time delivery for partner review
print("\n=== Streaming Response (Partner Review) ===")
stream = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "List the top 3 priorities for a digital transformation engagement at a Fortune 500 retailer, one per line."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100")),
    stream=True
)

collected_text = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        collected_text += token
        print(token, end="", flush=True)

print(f"\n\nFull collected text: {collected_text}")

### Demo 2: Structured Output with JSON Mode — Client Profile Extraction

By setting `response_format={"type": "json_object"}`, the model is guaranteed to return valid JSON. This is critical for consulting workflows where downstream systems must parse client data reliably. You **must** include the word "JSON" in your prompt when using this mode.

**McKinsey Scenario:** An Engagement Manager needs a structured client profile for a new healthcare engagement. JSON mode ensures the profile data is machine-readable for the firm's CRM and engagement tracking systems.

In [ ]:
# Demo 2 - Structured Output with JSON Mode (McKinsey Client Profile)

client = openai.OpenAI()

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey engagement assistant that outputs JSON."},
        {"role": "user", "content": "Create a client profile for a new engagement with Acme Healthcare Corp. Include: client_name, industry, annual_revenue_usd, number_of_employees, headquarters, key_challenges (list), and engagement_type."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw_json = response.choices[0].message.content
print("Raw JSON response:")
print(raw_json)

parsed = json.loads(raw_json)
print("\nParsed and formatted:")
print(json.dumps(parsed, indent=2))

print(f"\nClient      : {parsed.get('client_name', 'N/A')}")
print(f"Industry    : {parsed.get('industry', 'N/A')}")
print(f"Engagement  : {parsed.get('engagement_type', 'N/A')}")

### Demo 3: Function Calling — Consulting Research Tools

Function calling lets the model decide **when** and **how** to call external tools. In a consulting context, this enables an AI assistant to autonomously pull market data, run benchmarks, or look up competitor intelligence. The flow is:
1. Define tool schemas (e.g., `market_research`, `competitor_lookup`) and send them with the request
2. Model returns a `tool_calls` response
3. Execute the function locally (simulated data)
4. Send the result back for a final answer

**McKinsey Scenario:** An Associate asks the AI for market research on a specific industry. The model decides to call the `market_research` tool to retrieve structured market data.

In [ ]:
# Demo 3 - Function Calling / Tool Use (McKinsey Consulting Research)

client = openai.OpenAI()

tools = [
    {
        "type": "function",
        "function": {
            "name": "market_research",
            "description": "Get market research data for a specific industry sector",
            "parameters": {
                "type": "object",
                "properties": {
                    "industry": {"type": "string", "description": "The industry sector, e.g., 'healthcare', 'financial_services', 'energy'"},
                    "region": {"type": "string", "enum": ["north_america", "europe", "asia_pacific", "global"], "description": "Geographic region for the analysis"}
                },
                "required": ["industry"]
            }
        }
    }
]

def market_research(industry, region="global"):
    """Simulated market research function returning industry data."""
    market_data = {
        "healthcare": {"market_size_usd_bn": 8450, "cagr_pct": 7.9, "key_trend": "AI-driven diagnostics and personalized medicine"},
        "financial_services": {"market_size_usd_bn": 28500, "cagr_pct": 6.2, "key_trend": "Embedded finance and real-time payments"},
        "energy": {"market_size_usd_bn": 6700, "cagr_pct": 8.5, "key_trend": "Energy transition and grid modernization"},
    }
    data = market_data.get(industry, {"market_size_usd_bn": 5000, "cagr_pct": 5.0, "key_trend": "Digital transformation"})
    return json.dumps({"industry": industry, "region": region, "market_size_usd_bn": data["market_size_usd_bn"], "cagr_pct": data["cagr_pct"], "key_trend": data["key_trend"]})

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")

if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"Function called: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")
    args = json.loads(tool_call.function.arguments)
    result = market_research(**args)
    print(f"Function result: {result}")

    follow_up = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."},
            message,
            {"role": "tool", "tool_call_id": tool_call.id, "content": result}
        ],
        tools=tools
    )
    print(f"\nFinal response:\n{follow_up.choices[0].message.content}")
else:
    print(f"Direct response: {message.content}")

### Demo 4: Pydantic-Based Response Validation — Engagement Summary

Pydantic models let you define the **exact shape** of the data you expect, with type checking and constraints. In consulting, this ensures that engagement summaries, client assessments, and deliverable metadata conform to the firm's standards.

**McKinsey Scenario:** Validate a structured engagement summary for a recently completed strategy engagement. The Pydantic model enforces that all required fields (client name, industry, engagement type, satisfaction score, etc.) are present and correctly typed.

In [ ]:
# Demo 4 - Pydantic-Based Response Validation (McKinsey Engagement Summary)

client = openai.OpenAI()

class EngagementSummary(BaseModel):
    client_name: str = Field(description="Name of the client organization")
    industry: str = Field(description="Client industry sector")
    engagement_type: str = Field(description="Type of engagement, e.g., Strategy, M&A Due Diligence, Digital Transformation")
    duration_weeks: int = Field(description="Duration of the engagement in weeks")
    satisfaction_score: float = Field(ge=1.0, le=5.0, description="Client satisfaction score (1-5)")
    key_recommendation: str = Field(description="One-sentence primary recommendation")
    follow_on_opportunity: bool = Field(description="Whether there is a follow-on engagement opportunity")

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey engagement tracking system. Output your engagement summary as JSON with these exact fields: client_name (string), industry (string), engagement_type (string), duration_weeks (integer), satisfaction_score (float 1-5), key_recommendation (string), follow_on_opportunity (boolean)."},
        {"role": "user", "content": "Summarize the recently completed digital transformation engagement with Meridian Financial Group."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw = response.choices[0].message.content
print("Raw JSON:")
print(raw)

try:
    summary = EngagementSummary.model_validate_json(raw)
    print("\nValidated EngagementSummary:")
    print(f"  Client          : {summary.client_name}")
    print(f"  Industry        : {summary.industry}")
    print(f"  Engagement Type : {summary.engagement_type}")
    print(f"  Duration        : {summary.duration_weeks} weeks")
    print(f"  Satisfaction    : {summary.satisfaction_score}/5.0")
    print(f"  Recommendation  : {summary.key_recommendation}")
    print(f"  Follow-on       : {'Yes' if summary.follow_on_opportunity else 'No'}")
except Exception as e:
    print(f"\nValidation error: {e}")

### Demo 5: Building a Structured Data Extraction Pipeline — Engagement Team Extraction

This demo combines JSON mode and Pydantic into a reusable pipeline that extracts structured consulting team information from unstructured engagement notes. This pattern is the foundation for many agentic data-processing workflows at McKinsey — from staffing databases to knowledge management systems.

**McKinsey Scenario:** An engagement kickoff email contains team member details in unstructured text. We build a pipeline to extract each team member's name, role, email, practice area, and office location into structured data.

In [ ]:
# Demo 5 - Building a Structured Data Extraction Pipeline (McKinsey Engagement Team)

client = openai.OpenAI()

class TeamMember(BaseModel):
    name: str = Field(description="Full name of the team member")
    email: Optional[str] = Field(default=None, description="Email address if found")
    phone: Optional[str] = Field(default=None, description="Phone number if found")
    role: Optional[str] = Field(default=None, description="Consulting role: Partner, Associate Partner, Engagement Manager, Associate, Business Analyst")
    practice: Optional[str] = Field(default=None, description="McKinsey practice area if found")

def extract_team_members(text: str) -> List[TeamMember]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "Extract all consulting team member information from the text. Return a JSON object with a 'team_members' key containing a list. Each member should have: name, email (null if not found), phone (null if not found), role (null if not found), practice (null if not found)."},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )
    data = json.loads(response.choices[0].message.content)
    members = []
    for item in data.get("team_members", []):
        try:
            members.append(TeamMember(**item))
        except Exception as e:
            print(f"Skipping invalid team member: {e}")
    return members

sample_text = """Team, here is the staffing for the Apex Energy transformation engagement:

The engagement will be led by Partner Rajesh Gupta from our Houston office (Energy Practice).
Reach him at rajesh.gupta@mckinsey.com or (713) 555-0142.

Engagement Manager Sarah Okonkwo from the Chicago office will run day-to-day operations.
She is part of our Operations Practice. Her email is sarah.okonkwo@mckinsey.com.

We also have two Associates staffed: David Chen (Digital/McKinsey Digital, New York office,
david.chen@mckinsey.com) and Priya Sharma (Strategy & Corporate Finance, London office).

For analytics support, reach out to Business Analyst Tom Williams at tom.williams@mckinsey.com."""

print("Extracted engagement team:")
print("=" * 60)
members = extract_team_members(sample_text)
for i, member in enumerate(members, 1):
    print(f"\nTeam Member {i}:")
    print(f"  Name     : {member.name}")
    print(f"  Email    : {member.email or 'N/A'}")
    print(f"  Phone    : {member.phone or 'N/A'}")
    print(f"  Role     : {member.role or 'N/A'}")
    print(f"  Practice : {member.practice or 'N/A'}")

---
## Tasks -- Full Solutions

Below are the complete, working solutions for all four student tasks. Each solution is preceded by an explanation of the approach. All tasks use McKinsey consulting scenarios.

### Task 1: Build a Structured Competitive Assessment Extractor Using JSON Mode

**Approach:** The solution uses JSON mode with a system message that defines the expected schema for extracting competitive intelligence from an M&A due diligence briefing. We extract key entities relevant to consulting: companies, executives, financial metrics, and strategic actions. The key teaching point is how the system message acts as a schema definition for MECE (Mutually Exclusive, Collectively Exhaustive) data extraction.

In [ ]:
# Task 1 - SOLUTION: Build a Structured Competitive Assessment Extractor Using JSON Mode

def extract_competitive_intelligence(briefing_text):
    """Extract competitive intelligence entities from an M&A due diligence briefing using JSON mode."""
    client = openai.OpenAI()

    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": (
                "You are a McKinsey M&A due diligence analyst. Extract competitive intelligence from the briefing text. "
                "Return a JSON object with these keys: "
                "companies (list of strings — all company names mentioned), "
                "executives (list of strings — all named executives with their titles), "
                "financial_metrics (list of strings — all revenue, valuation, or financial figures mentioned), "
                "strategic_actions (list of strings — mergers, acquisitions, partnerships, or strategic moves mentioned), "
                "industries (list of strings — all industry sectors referenced). "
                "If a category has no entities, use an empty list."
            )},
            {"role": "user", "content": briefing_text}
        ],
        response_format={"type": "json_object"},
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "400"))
    )

    return json.loads(response.choices[0].message.content)


# Test with an M&A due diligence briefing
sample_briefing = (
    "In Q3 2024, Meridian Health Systems (CEO: Dr. Amanda Foster) announced a $2.3B acquisition "
    "of Pacific Diagnostics, a move that positions them as the third-largest diagnostic network in "
    "North America. Goldman Sachs advised on the deal. Meanwhile, competitor UnitedHealth Group "
    "(CEO: Andrew Witty) reported $92.9B in quarterly revenue and signaled interest in expanding "
    "its Optum Health division through further acquisitions in the telehealth space. "
    "McKinsey Partner Lisa Chen is leading the due diligence workstream for a private equity client "
    "evaluating a potential counter-bid. The target company operates across healthcare and life sciences."
)
entities = extract_competitive_intelligence(sample_briefing)
print("Extracted Competitive Intelligence:")
print(json.dumps(entities, indent=2))

### Task 2: Implement a Financial Analysis Tool with Function Calling

**Approach:** We define a `financial_analysis` tool with parameters for analysis type (valuation, profitability, leverage, liquidity) and a company identifier. The `ask_financial_question` function handles the full loop: initial request, detecting the tool call, executing the function, and sending the result back for a natural-language answer. This mirrors how a McKinsey Associate would query financial databases during an engagement.

In [ ]:
# Task 2 - SOLUTION: Implement a Financial Analysis Tool with Function Calling

def financial_analysis(analysis_type, company):
    """Perform a financial analysis on a company (simulated data)."""
    company_data = {
        "Meridian Health": {
            "valuation": {"ev_ebitda": 14.2, "pe_ratio": 22.5, "market_cap_bn": 18.7},
            "profitability": {"gross_margin_pct": 42.3, "ebitda_margin_pct": 18.1, "net_margin_pct": 8.9},
            "leverage": {"debt_to_equity": 1.4, "net_debt_ebitda": 3.2, "interest_coverage": 5.8},
            "liquidity": {"current_ratio": 1.8, "quick_ratio": 1.2, "cash_bn": 2.1}
        },
        "Apex Energy": {
            "valuation": {"ev_ebitda": 8.5, "pe_ratio": 15.3, "market_cap_bn": 42.1},
            "profitability": {"gross_margin_pct": 55.7, "ebitda_margin_pct": 32.4, "net_margin_pct": 14.2},
            "leverage": {"debt_to_equity": 0.8, "net_debt_ebitda": 1.9, "interest_coverage": 8.3},
            "liquidity": {"current_ratio": 2.1, "quick_ratio": 1.6, "cash_bn": 5.4}
        }
    }
    data = company_data.get(company, {}).get(analysis_type)
    if data is None:
        return json.dumps({"error": f"No {analysis_type} data available for {company}"})
    return json.dumps({"company": company, "analysis_type": analysis_type, **data})


def ask_financial_question(question):
    """Send a financial analysis question and handle function calling."""
    client = openai.OpenAI()

    fin_tool = [{
        "type": "function",
        "function": {
            "name": "financial_analysis",
            "description": "Perform a financial analysis on a company. Returns key financial metrics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "analysis_type": {"type": "string", "enum": ["valuation", "profitability", "leverage", "liquidity"],
                                      "description": "Type of financial analysis to perform"},
                    "company": {"type": "string", "description": "Company name to analyze"}
                },
                "required": ["analysis_type", "company"]
            }
        }
    }]

    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": question}],
        tools=fin_tool,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        result = financial_analysis(**args)

        follow_up = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            messages=[
                {"role": "user", "content": question},
                message,
                {"role": "tool", "tool_call_id": tool_call.id, "content": str(result)}
            ],
            tools=fin_tool
        )
        return follow_up.choices[0].message.content

    return message.content


# Test
print(ask_financial_question("What is the valuation profile of Meridian Health?"))
print()
print(ask_financial_question("How profitable is Apex Energy? Give me the margin analysis."))

### Task 3: Create a Multi-Tool Consulting Agent with Automatic Tool Dispatch

**Approach:** We define three consulting tools (market_research, financial_analysis, benchmarking), implement each as a simulated function, and use a dispatch dictionary to route tool calls. The agent handles both single and multiple tool calls, plus direct responses when no tool is needed. This mirrors how a McKinsey engagement team would leverage multiple knowledge sources during a strategy engagement.

In [ ]:
# Task 3 - SOLUTION: Create a Multi-Tool Consulting Agent with Automatic Tool Dispatch

# Tool implementations
def tool_market_research(industry, region="global"):
    market_data = {
        "healthcare": {"market_size_usd_bn": 8450, "cagr_pct": 7.9, "top_player": "UnitedHealth Group", "key_trend": "AI-driven diagnostics"},
        "financial_services": {"market_size_usd_bn": 28500, "cagr_pct": 6.2, "top_player": "JPMorgan Chase", "key_trend": "Embedded finance"},
        "energy": {"market_size_usd_bn": 6700, "cagr_pct": 8.5, "top_player": "ExxonMobil", "key_trend": "Energy transition"}
    }
    data = market_data.get(industry, {"market_size_usd_bn": 5000, "cagr_pct": 5.0, "top_player": "N/A", "key_trend": "Digital transformation"})
    return json.dumps({"industry": industry, "region": region, **data})

def tool_financial_analysis(company, metric):
    company_metrics = {
        "Meridian Health": {"revenue_bn": 12.4, "ebitda_margin_pct": 18.1, "debt_to_equity": 1.4, "roe_pct": 15.3},
        "Apex Energy": {"revenue_bn": 38.7, "ebitda_margin_pct": 32.4, "debt_to_equity": 0.8, "roe_pct": 22.1},
        "TechNova Inc": {"revenue_bn": 5.2, "ebitda_margin_pct": 24.7, "debt_to_equity": 0.3, "roe_pct": 28.9}
    }
    data = company_metrics.get(company, {"revenue_bn": 0, "ebitda_margin_pct": 0, "debt_to_equity": 0, "roe_pct": 0})
    return json.dumps({"company": company, "metric": metric, "value": data.get(metric, "N/A")})

def tool_benchmarking(company, peer_group):
    benchmarks = {
        "healthcare": {"avg_ebitda_margin": 16.5, "avg_revenue_growth": 6.8, "avg_roe": 14.2},
        "energy": {"avg_ebitda_margin": 28.3, "avg_revenue_growth": 4.5, "avg_roe": 18.7},
        "technology": {"avg_ebitda_margin": 22.1, "avg_revenue_growth": 12.3, "avg_roe": 25.4}
    }
    data = benchmarks.get(peer_group, {"avg_ebitda_margin": 15.0, "avg_revenue_growth": 5.0, "avg_roe": 12.0})
    return json.dumps({"company": company, "peer_group": peer_group, **data})

# Tool definitions
tools = [
    {"type": "function", "function": {"name": "market_research", "description": "Get market research data for an industry sector including market size, growth rate, and key trends",
        "parameters": {"type": "object", "properties": {"industry": {"type": "string", "description": "Industry sector: healthcare, financial_services, energy, technology, retail"},
            "region": {"type": "string", "enum": ["north_america", "europe", "asia_pacific", "global"]}}, "required": ["industry"]}}},
    {"type": "function", "function": {"name": "financial_analysis", "description": "Get a specific financial metric for a company",
        "parameters": {"type": "object", "properties": {"company": {"type": "string", "description": "Company name"},
            "metric": {"type": "string", "enum": ["revenue_bn", "ebitda_margin_pct", "debt_to_equity", "roe_pct"]}}, "required": ["company", "metric"]}}},
    {"type": "function", "function": {"name": "benchmarking", "description": "Benchmark a company against its industry peer group averages",
        "parameters": {"type": "object", "properties": {"company": {"type": "string", "description": "Company name to benchmark"},
            "peer_group": {"type": "string", "description": "Industry peer group: healthcare, energy, technology, financial_services"}}, "required": ["company", "peer_group"]}}}
]

# Dispatch table
dispatch = {
    "market_research": lambda args: tool_market_research(args["industry"], args.get("region", "global")),
    "financial_analysis": lambda args: tool_financial_analysis(args["company"], args["metric"]),
    "benchmarking": lambda args: tool_benchmarking(args["company"], args["peer_group"])
}

def multi_tool_consulting_agent(user_message):
    client = openai.OpenAI()
    messages = [
        {"role": "system", "content": "You are a McKinsey consulting research assistant. Use the available tools to answer questions about markets, company financials, and industry benchmarks."},
        {"role": "user", "content": user_message}
    ]

    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=messages, tools=tools, tool_choice="auto"
    )
    message = response.choices[0].message

    if message.tool_calls:
        messages.append(message)
        for tc in message.tool_calls:
            args = json.loads(tc.function.arguments)
            result = dispatch[tc.function.name](args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        final = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=messages, tools=tools
        )
        return final.choices[0].message.content

    return message.content


# Test
print("--- Market Research Query ---")
print(multi_tool_consulting_agent("What is the market size and growth rate for the healthcare industry?"))
print("\n--- Financial Analysis Query ---")
print(multi_tool_consulting_agent("What is the EBITDA margin for Apex Energy?"))
print("\n--- Benchmarking Query ---")
print(multi_tool_consulting_agent("How does Meridian Health compare to its healthcare peer group?"))

### Task 4: Build a Robust Consulting API Client with Retries, Validation, and Streaming

**Approach:** The `RobustConsultingClient` class wraps the OpenAI client with: (1) exponential backoff retry logic for production reliability, (2) optional Pydantic validation via `model_validate_json()` for structured consulting outputs, (3) streaming support for real-time partner briefings, and (4) cumulative token usage tracking for engagement cost management. This is a production-ready pattern for McKinsey's AI-powered consulting tools.

In [ ]:
# Task 4 - SOLUTION: Build a Robust Consulting API Client with Retries, Validation, and Streaming

class RobustConsultingClient:
    def __init__(self, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), max_retries=3):
        self.client = openai.OpenAI()
        self.model = model
        self.max_retries = max_retries
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_calls = 0

    def call(self, messages, response_model=None, stream=False, **kwargs):
        """Make a robust API call with retries and optional Pydantic validation."""
        for attempt in range(self.max_retries):
            try:
                if stream:
                    return self._stream_call(messages, **kwargs)

                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    response_format={"type": "json_object"} if response_model else None,
                    **kwargs
                )

                self.total_prompt_tokens += response.usage.prompt_tokens
                self.total_completion_tokens += response.usage.completion_tokens
                self.total_calls += 1

                content = response.choices[0].message.content

                if response_model:
                    return response_model.model_validate_json(content)
                return content

            except Exception as e:
                wait = 2 ** attempt
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
                if attempt < self.max_retries - 1:
                    time.sleep(wait)
                else:
                    raise

    def _stream_call(self, messages, **kwargs):
        stream = self.client.chat.completions.create(
            model=self.model, messages=messages, stream=True, **kwargs
        )
        collected = ""
        for chunk in stream:
            if chunk.choices[0].delta.content:
                token = chunk.choices[0].delta.content
                collected += token
                print(token, end="", flush=True)
        print()
        self.total_calls += 1
        return collected

    def get_usage_stats(self):
        return {
            "total_calls": self.total_calls,
            "total_prompt_tokens": self.total_prompt_tokens,
            "total_completion_tokens": self.total_completion_tokens,
            "total_tokens": self.total_prompt_tokens + self.total_completion_tokens
        }


# Test non-streaming with Pydantic validation — McKinsey Client Profile
consulting_client = RobustConsultingClient()

class ClientProfile(BaseModel):
    client_name: str
    industry: str
    engagement_type: str
    annual_revenue_bn: float

result = consulting_client.call(
    messages=[
        {"role": "system", "content": "Output JSON with fields: client_name, industry, engagement_type, annual_revenue_bn"},
        {"role": "user", "content": "Create a client profile for Apex Energy, an oil and gas company with $38.7B revenue. We are doing an energy transition strategy engagement."}
    ],
    response_model=ClientProfile
)
print(f"Validated: {result.client_name} ({result.industry}) - {result.engagement_type}, ${result.annual_revenue_bn}B revenue")
print(f"Usage: {consulting_client.get_usage_stats()}")

# Test streaming — CEO Briefing
print("\nStreaming CEO Briefing:")
consulting_client.call(
    messages=[{"role": "user", "content": "Provide a 2-sentence executive summary of the key recommendation for a digital transformation at a mid-size retailer."}],
    stream=True,
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)
print(f"Usage after streaming: {consulting_client.get_usage_stats()}")

---
## Summary

In this session students learned production-grade OpenAI API engineering skills applied to McKinsey consulting workflows:

1. **API Deep Dive** -- Streaming CEO briefings for real-time partner review, tracking token usage for engagement cost management, and understanding finish reasons.
2. **JSON Mode** -- Using `response_format={"type": "json_object"}` to generate reliable, parseable structured data (client profiles, market data) from the model.
3. **Function Calling** -- Defining consulting tools (market_research, financial_analysis, benchmarking) that the model can invoke, handling the request-execute-respond loop that powers agentic tool use.
4. **Pydantic Validation** -- Using Pydantic models (EngagementSummary, ClientProfile) to validate and parse LLM outputs with type safety and automatic error detection.
5. **Extraction Pipelines** -- Combining JSON mode and Pydantic into reusable pipelines that extract structured consulting data (engagement teams, competitive intelligence) from unstructured text.

**Instructor Notes:**
- Emphasize that structured outputs are the bridge between LLMs and consulting workflows -- without them, agents cannot act on LLM decisions to populate CRM systems, generate deliverables, or trigger downstream analyses.
- For Task 2, discuss how the model sometimes tries to answer financial questions directly -- show that `tool_choice="required"` forces tool use.
- For Task 3, highlight how the multi-tool pattern mirrors a real consulting engagement where Associates pull from multiple knowledge sources (market data, financials, benchmarks) to build a MECE analysis.
- For Task 4, discuss production concerns: rate limits, cost tracking across engagements, and graceful degradation.

**Up Next:** In Session 2, we will move from raw API calls to LangChain, learning how to compose modular chains, integrate tools, and build retrieval-augmented generation (RAG) systems for consulting knowledge management.